# Part 2 — Cough Detection

**Pipeline:** `Audio → Energy Segmentation → HeAR Embedding → 2-Stage Classification`

- **Stage 1**: 3-class (dry / wet / none) — detect cough segments
- **Stage 2**: 2-class (dry / wet) — reclassify cough-positive segments
- **Final**: Majority vote across segments → file-level prediction

This pipeline enables the Care AI nurse (Part 3) to analyze respiratory symptoms  
from audio captured during video calls.

---

This notebook is part of the **MedGemma Clinical Trial Engine** pipeline:

```
Part 1  Visual AE Detection ─── MedGemma 1.5 + MedSigLIP (image → AE classification)
Part 2  Cough Detection ──────── HeAR + 2-Stage Classifier (audio → cough type)
Part 3  Care AI Conversation ── MedGemma-4B as virtual nurse (multi-turn dialogue → AE detection)
    ↓
Part 4  Rule Set Generation ─── 10 biomedical DBs → LLM synthesis → simulation parameters
Part 5  Simulation Pipeline ─── Hazard functions + LLM enrichment → synthetic clinical trial data
    ↓
Part 6  Anti-Hallucination ──── RLFR fine-tuning to reduce fabrication in medical text
Part 7  Doc Agent ──────────────  CRF data → MedWatch 3500A pharmacovigilance reports
```


## 0. Setup & Configuration

In [ ]:
# Run once to install dependencies (requires TensorFlow)
!pip install -q tensorflow tensorflow-hub librosa soundfile joblib scikit-learn numpy matplotlib huggingface-hub

In [ ]:
# [최초 1회만 실행] 커널 재시작 — TF GPU 초기화 문제 방지
# 실행하면 커널이 재시작됩니다. 재시작 후 이 셀은 건너뛰고 아래 셀부터 실행하세요.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path
from collections import Counter

# === GPU setup — MUST happen before any TF import ===
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("GPU_ID", "6")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# Pre-allocate only needed memory instead of grabbing entire GPU
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
try:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError:
    pass  # already initialized — TF_FORCE_GPU_ALLOW_GROWTH handles it
print(f"TF GPUs: {tf.config.list_physical_devices('GPU')}")

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import IPython.display as ipd

warnings.filterwarnings('ignore')

# === CONFIG ===
SR = 16000

ROOT = Path("/data2/workspace/ClinicalTrialEngine")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

print(f"ROOT: {ROOT}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load Models

In [ ]:
# --- 1a. HeAR model (TF + GPU already configured in Setup cell) ---
from huggingface_hub import snapshot_download

hear_dir = snapshot_download("google/hear")
hear_model = tf.saved_model.load(hear_dir)
hear_serving = hear_model.signatures["serving_default"]
print("HeAR model loaded.")

In [ ]:
# --- 1b. Classifiers ---
import joblib

# Stage 1: 3-class (dry / wet / none)
HEAR_MIXED_MODEL  = os.environ.get("HEAR_MIXED_MODEL",  str(ROOT / "data" / "multimodal" / "hear_mixed_model"))
model_dir_3c = Path(HEAR_MIXED_MODEL)
clf_3c = joblib.load(model_dir_3c / "classifier.joblib")
le_3c  = joblib.load(model_dir_3c / "label_encoder.joblib")
print(f"Stage 1 (3-class): {le_3c.classes_}  — {type(clf_3c).__name__}")

# Stage 2: 2-class (dry / wet)
HEAR_COUGH_MODEL  = os.environ.get("HEAR_COUGH_MODEL",  str(ROOT / "data" / "multimodal" / "hear_cough_only_model"))
model_dir_2c = Path(HEAR_COUGH_MODEL)
clf_2c = joblib.load(model_dir_2c / "classifier.joblib")
le_2c  = joblib.load(model_dir_2c / "label_encoder.joblib")
print(f"Stage 2 (2-class): {le_2c.classes_}  — {type(clf_2c).__name__}")

In [ ]:
# --- 1c. Segmentation function ---
from cough_detection.segmentation import segment_cough
print("segment_cough loaded from detect-segment-cough.")

## 2. Select Audio File

In [ ]:
# ========== CHANGE THIS PATH TO YOUR AUDIO FILE ==========
AUDIO_PATH = ROOT / "data" / "multimodal" / "generated_voices_v4" / "wet" / "wet_02_frequent_female.wav"
# ==========================================================

# Load audio
audio, sr = librosa.load(str(AUDIO_PATH), sr=SR)
duration = len(audio) / sr

print(f"File:     {AUDIO_PATH.name}")
print(f"Duration: {duration:.2f}s")
print(f"SR:       {sr} Hz")
print(f"Samples:  {len(audio):,}")

# Play audio
ipd.display(ipd.Audio(audio, rate=sr))

In [ ]:
# Load GT (if manifest exists)
manifest_path = AUDIO_PATH.parent.parent / "manifest.json"
gt_info = None
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    fname_key = f"{AUDIO_PATH.parent.name}/{AUDIO_PATH.name}"
    for s in manifest["samples"]:
        if s["filename"] == fname_key:
            gt_info = s
            break

if gt_info:
    print(f"GT Label:       {gt_info['label']}")
    print(f"Description:    {gt_info['description']}")
    print(f"Cough Inserted: {gt_info['cough_inserted']}")
    if gt_info.get("metadata", {}).get("cough_timestamps"):
        print(f"\nGT Cough Timestamps:")
        for ct in gt_info["metadata"]["cough_timestamps"]:
            print(f"  {ct['start_sec']:.3f}s ~ {ct['end_sec']:.3f}s  ({ct['duration_sec']:.1f}s, {ct['cough_type']})")
else:
    print("No manifest/GT found for this file.")

## 3. Waveform & Spectrogram Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
time_axis = np.arange(len(audio)) / sr

# Waveform
axes[0].plot(time_axis, audio, linewidth=0.3, color='steelblue')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Waveform — {AUDIO_PATH.name}')

# Mark GT cough regions
if gt_info and gt_info.get("metadata", {}).get("cough_timestamps"):
    for ct in gt_info["metadata"]["cough_timestamps"]:
        axes[0].axvspan(ct['start_sec'], ct['end_sec'], alpha=0.2, color='red', label='GT cough')
    # Deduplicate legend
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    axes[0].legend(by_label.values(), by_label.keys(), loc='upper right')

# Spectrogram
S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, fmax=8000)
S_dB = librosa.power_to_db(S, ref=np.max)
librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], cmap='magma')
axes[1].set_title('Mel Spectrogram')

plt.tight_layout()
plt.show()

## 4. Energy-Based Segmentation (Step 1)

In [ ]:
# Run energy-based hysteresis segmentation
segments, mask = segment_cough(audio, sr, cough_padding=0)

# Extract segment timestamps
changes = np.diff(mask.astype(int))
starts = np.where(changes == 1)[0] + 1
ends   = np.where(changes == -1)[0] + 1

# Energy thresholds (same as segmentation.py uses)
rms_global = np.sqrt(np.mean(np.square(audio)))
th_low  = 0.1 * rms_global
th_high = 2.0 * rms_global

print(f"Global RMS:    {rms_global:.4f}")
print(f"Threshold Low: {th_low:.4f}  (0.1 × RMS)")
print(f"Threshold High:{th_high:.4f}  (2.0 × RMS)")
print(f"Segments Found: {len(segments)}")
print()

# Segment table
print(f"{'Seg':>3}  {'Start':>7}  {'End':>7}  {'Dur':>6}  {'RMS':>7}  {'Samples':>8}")
print("-" * 50)
for i, (s, e) in enumerate(zip(starts, ends)):
    if i >= len(segments):
        break
    seg_rms = np.sqrt(np.mean(segments[i] ** 2))
    print(f"{i:>3}  {s/sr:>7.3f}  {e/sr:>7.3f}  {(e-s)/sr:>5.3f}s  {seg_rms:>7.4f}  {len(segments[i]):>8}")

In [ ]:
# Visualize segmentation on waveform
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(time_axis, audio, linewidth=0.3, color='gray', alpha=0.6)

# Highlight energy thresholds
ax.axhline(y=th_high, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label=f'Th_high ({th_high:.4f})')
ax.axhline(y=-th_high, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(y=th_low, color='orange', linestyle=':', linewidth=0.8, alpha=0.5, label=f'Th_low ({th_low:.4f})')
ax.axhline(y=-th_low, color='orange', linestyle=':', linewidth=0.8, alpha=0.5)

# Detected segments
colors = plt.cm.tab10(np.linspace(0, 1, max(len(starts), 1)))
for i, (s, e) in enumerate(zip(starts, ends)):
    if i >= len(segments):
        break
    ax.axvspan(s/sr, e/sr, alpha=0.3, color=colors[i % len(colors)])
    ax.text((s/sr + e/sr) / 2, ax.get_ylim()[1] * 0.9, f'S{i}',
            ha='center', va='top', fontsize=8, fontweight='bold')

# GT cough overlay
if gt_info and gt_info.get("metadata", {}).get("cough_timestamps"):
    for ct in gt_info["metadata"]["cough_timestamps"]:
        ax.axvspan(ct['start_sec'], ct['end_sec'], alpha=0.15, color='red',
                   ymin=0.0, ymax=0.1)
    ax.plot([], [], color='red', alpha=0.3, linewidth=8, label='GT cough region')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Energy Segmentation — {len(segments)} segments detected')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 5. HeAR Embedding Extraction (Step 2)

In [ ]:
TARGET_LEN = 2 * SR  # 2 seconds = 32000 samples

def get_embedding(audio_segment):
    """Extract HeAR 512-d embedding from a segment."""
    if len(audio_segment) < TARGET_LEN:
        padded = np.pad(audio_segment, (0, TARGET_LEN - len(audio_segment)))
    else:
        padded = audio_segment[:TARGET_LEN]
    return hear_serving(x=tf.constant(padded.reshape(1, -1), dtype=tf.float32))["output_0"].numpy()

# Extract embeddings for all segments
embeddings = []
for i, seg in enumerate(segments):
    emb = get_embedding(seg)
    embeddings.append(emb)
    print(f"Seg {i}: {len(seg)} samples → embedding shape {emb.shape}, "
          f"norm={np.linalg.norm(emb):.2f}, mean={emb.mean():.4f}")

print(f"\nTotal: {len(embeddings)} embeddings extracted (512-d each)")

In [ ]:
# Visualize embeddings as heatmap
if len(embeddings) > 0:
    emb_matrix = np.vstack(embeddings)
    fig, ax = plt.subplots(figsize=(16, max(2, len(embeddings) * 0.4)))
    im = ax.imshow(emb_matrix, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    ax.set_xlabel('Embedding Dimension (512-d)')
    ax.set_ylabel('Segment')
    ax.set_yticks(range(len(embeddings)))
    ax.set_yticklabels([f'S{i} ({starts[i]/sr:.1f}s)' for i in range(len(embeddings))])
    ax.set_title('HeAR Embeddings per Segment')
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

## 6. Stage 1 — 3-Class Classification (dry / wet / none)

In [ ]:
stage1_results = []

print(f"{'Seg':>3}  {'Time':>12}  {'Label':<6}  {'none':>6}  {'dry':>6}  {'wet':>6}")
print("-" * 55)

for i, emb in enumerate(embeddings):
    pred = clf_3c.predict(emb)[0]
    proba = clf_3c.predict_proba(emb)[0]
    label = le_3c.inverse_transform([pred])[0]
    prob_dict = {c: round(float(p), 3) for c, p in zip(le_3c.classes_, proba)}

    stage1_results.append({
        "index": i,
        "start": round(starts[i] / sr, 3),
        "end": round(ends[i] / sr, 3),
        "label": label,
        "probabilities": prob_dict,
    })

    marker = "<<" if label != "none" else ""
    print(f"{i:>3}  {starts[i]/sr:>5.2f}-{ends[i]/sr:<5.2f}s  {label:<6}  "
          f"{prob_dict.get('none',0):>6.1%}  {prob_dict.get('dry',0):>6.1%}  {prob_dict.get('wet',0):>6.1%}  {marker}")

n_cough = sum(1 for r in stage1_results if r['label'] != 'none')
n_none  = sum(1 for r in stage1_results if r['label'] == 'none')
print(f"\nSummary: {n_cough} cough segments, {n_none} non-cough segments")

In [ ]:
# Visualize Stage 1 probabilities
if len(stage1_results) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))

    seg_indices = range(len(stage1_results))
    none_probs = [r['probabilities'].get('none', 0) for r in stage1_results]
    dry_probs  = [r['probabilities'].get('dry', 0) for r in stage1_results]
    wet_probs  = [r['probabilities'].get('wet', 0) for r in stage1_results]

    bar_width = 0.6
    ax.bar(seg_indices, none_probs, bar_width, label='none', color='#4CAF50', alpha=0.8)
    ax.bar(seg_indices, dry_probs, bar_width, bottom=none_probs, label='dry', color='#FF9800', alpha=0.8)
    ax.bar(seg_indices, wet_probs, bar_width,
           bottom=[n + d for n, d in zip(none_probs, dry_probs)], label='wet', color='#2196F3', alpha=0.8)

    ax.set_xticks(seg_indices)
    ax.set_xticklabels([f'S{r["index"]}\n{r["start"]:.1f}s' for r in stage1_results], fontsize=8)
    ax.set_ylabel('Probability')
    ax.set_title('Stage 1: 3-Class Probabilities per Segment')
    ax.legend(loc='upper right')
    ax.set_ylim(0, 1.05)

    # Mark cough segments
    for i, r in enumerate(stage1_results):
        if r['label'] != 'none':
            ax.annotate(r['label'], (i, 1.02), ha='center', fontsize=8, fontweight='bold', color='red')

    plt.tight_layout()
    plt.show()

## 7. Stage 2 — 2-Class Re-classification (dry / wet)

In [ ]:
stage2_results = []

print("Only cough segments from Stage 1 are re-classified:")
print(f"{'Seg':>3}  {'Time':>12}  {'S1 Label':<9}  {'S2 Label':<9}  {'dry':>6}  {'wet':>6}  {'Final none':>10}  {'Final dry':>10}  {'Final wet':>10}")
print("-" * 95)

for i, (emb, s1) in enumerate(zip(embeddings, stage1_results)):
    if s1['label'] == 'none':
        # Pass through as-is
        stage2_results.append({
            **s1,
            "stage": "1-none",
            "final_label": "none",
            "final_probs": s1['probabilities'],
        })
        print(f"{i:>3}  {s1['start']:>5.2f}-{s1['end']:<5.2f}s  {'none':<9}  {'—':<9}  {'—':>6}  {'—':>6}  "
              f"{s1['probabilities'].get('none',0):>10.1%}  {s1['probabilities'].get('dry',0):>10.1%}  {s1['probabilities'].get('wet',0):>10.1%}")
    else:
        # Stage 2 re-classification
        pred_2c = clf_2c.predict(emb)[0]
        proba_2c = clf_2c.predict_proba(emb)[0]
        label_2c = le_2c.inverse_transform([pred_2c])[0]
        prob_2c = {c: round(float(p), 3) for c, p in zip(le_2c.classes_, proba_2c)}

        # Merge: none from Stage1, dry/wet from Stage2 scaled by cough probability
        none_prob = s1['probabilities'].get('none', 0.0)
        cough_prob = 1.0 - none_prob
        final_probs = {
            'none': none_prob,
            'dry': round(cough_prob * prob_2c.get('dry', 0.0), 3),
            'wet': round(cough_prob * prob_2c.get('wet', 0.0), 3),
        }

        stage2_results.append({
            **s1,
            "stage": "2-cough",
            "stage2_label": label_2c,
            "stage2_probs": prob_2c,
            "final_label": label_2c,
            "final_probs": final_probs,
        })
        print(f"{i:>3}  {s1['start']:>5.2f}-{s1['end']:<5.2f}s  {s1['label']:<9}  {label_2c:<9}  "
              f"{prob_2c.get('dry',0):>6.1%}  {prob_2c.get('wet',0):>6.1%}  "
              f"{final_probs['none']:>10.1%}  {final_probs['dry']:>10.1%}  {final_probs['wet']:>10.1%}")

In [ ]:
# Visualize Stage 2 — only cough segments
cough_s2 = [r for r in stage2_results if r['stage'] == '2-cough']

if cough_s2:
    fig, ax = plt.subplots(figsize=(max(6, len(cough_s2) * 1.5), 4))

    x_pos = range(len(cough_s2))
    dry_vals = [r['stage2_probs']['dry'] for r in cough_s2]
    wet_vals = [r['stage2_probs']['wet'] for r in cough_s2]

    bar_w = 0.35
    ax.bar([x - bar_w/2 for x in x_pos], dry_vals, bar_w, label='dry', color='#FF9800')
    ax.bar([x + bar_w/2 for x in x_pos], wet_vals, bar_w, label='wet', color='#2196F3')

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"S{r['index']}\n{r['start']:.1f}s" for r in cough_s2], fontsize=9)
    ax.set_ylabel('Probability')
    ax.set_title('Stage 2: dry vs wet Re-classification (cough segments only)')
    ax.legend()
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim(0, 1.05)

    for i, r in enumerate(cough_s2):
        winner = r['final_label']
        ax.annotate(winner, (i, max(dry_vals[i], wet_vals[i]) + 0.03),
                    ha='center', fontsize=9, fontweight='bold',
                    color='#FF9800' if winner == 'dry' else '#2196F3')

    plt.tight_layout()
    plt.show()
else:
    print("No cough segments detected — Stage 2 skipped.")

## 8. Final Prediction (Majority Vote)

In [ ]:
# Majority vote among cough segments
cough_labels = [r['final_label'] for r in stage2_results if r['final_label'] in ('dry', 'wet')]

if cough_labels:
    vote_counts = Counter(cough_labels)
    predicted = vote_counts.most_common(1)[0][0]
    print("Cough segment votes:")
    for label, count in vote_counts.most_common():
        pct = count / len(cough_labels) * 100
        bar = '#' * (count * 4)
        print(f"  {label:<5}: {count} ({pct:.0f}%)  {bar}")
else:
    predicted = "none"
    print("No cough segments detected.")

gt_label = gt_info['label'] if gt_info else '?'
correct = predicted == gt_label

print(f"\n{'='*40}")
print(f"  Ground Truth:  {gt_label}")
print(f"  Prediction:    {predicted}")
print(f"  Result:        {'CORRECT' if correct else 'WRONG'}")
print(f"{'='*40}")

## 9. Full Timeline Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1, 1]})

# --- Top: Waveform + segments + GT ---
ax = axes[0]
ax.plot(time_axis, audio, linewidth=0.3, color='gray', alpha=0.5)

color_map = {'none': '#4CAF50', 'dry': '#FF9800', 'wet': '#2196F3'}

for r in stage2_results:
    c = color_map.get(r['final_label'], 'gray')
    ax.axvspan(r['start'], r['end'], alpha=0.4, color=c)
    ax.text((r['start'] + r['end']) / 2, ax.get_ylim()[1] * 0.85,
            f"{r['final_label']}", ha='center', fontsize=7, fontweight='bold', color=c)

if gt_info and gt_info.get("metadata", {}).get("cough_timestamps"):
    for ct in gt_info["metadata"]["cough_timestamps"]:
        ax.axvspan(ct['start_sec'], ct['end_sec'], alpha=0.1, color='red', ymin=0, ymax=0.15)
        ax.text((ct['start_sec'] + ct['end_sec'])/2, ax.get_ylim()[0] * 0.9,
                f"GT:{ct['cough_type']}", ha='center', fontsize=7, color='red')

patches = [mpatches.Patch(color=c, alpha=0.4, label=l) for l, c in color_map.items()]
if gt_info and gt_info.get("metadata", {}).get("cough_timestamps"):
    patches.append(mpatches.Patch(color='red', alpha=0.15, label='GT cough'))
ax.legend(handles=patches, loc='upper right', fontsize=8)
ax.set_ylabel('Amplitude')
ax.set_title(f'Pipeline Result: GT={gt_label} → Pred={predicted} ({"CORRECT" if correct else "WRONG"})')

# --- Middle: Stage 1 none probability ---
ax = axes[1]
for r in stage2_results:
    mid = (r['start'] + r['end']) / 2
    w = r['end'] - r['start']
    none_p = r['final_probs'].get('none', 0)
    c = '#4CAF50' if none_p > 0.5 else '#E53935'
    ax.bar(mid, none_p, width=w, color=c, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.text(mid, none_p + 0.02, f'{none_p:.0%}', ha='center', fontsize=7)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('P(none)')
ax.set_ylim(0, 1.15)
ax.set_title('Stage 1: Cough Detection (green=speech, red=cough)')

# --- Bottom: Stage 2 dry/wet split ---
ax = axes[2]
for r in stage2_results:
    mid = (r['start'] + r['end']) / 2
    w = r['end'] - r['start']
    if r['stage'] == '2-cough':
        dry_p = r['stage2_probs']['dry']
        wet_p = r['stage2_probs']['wet']
        ax.bar(mid, dry_p, width=w, color='#FF9800', alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.bar(mid, -wet_p, width=w, color='#2196F3', alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.text(mid, dry_p + 0.03, f'd:{dry_p:.0%}', ha='center', fontsize=7, color='#E65100')
        ax.text(mid, -wet_p - 0.08, f'w:{wet_p:.0%}', ha='center', fontsize=7, color='#0D47A1')
    else:
        ax.bar(mid, 0.02, width=w, color='#4CAF50', alpha=0.3)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('dry(+) / wet(-)')
ax.set_ylim(-1.15, 1.15)
ax.set_xlabel('Time (s)')
ax.set_title('Stage 2: dry vs wet (cough segments only)')

plt.tight_layout()
plt.show()

## 10. Listen to Individual Segments

In [ ]:
# Play each segment with its classification
for i, (seg, r) in enumerate(zip(segments, stage2_results)):
    label = r['final_label']
    probs = r['final_probs']
    prob_str = "  ".join(f"{k}={v:.1%}" for k, v in probs.items())

    icon = {"none": "[speech]", "dry": "[DRY cough]", "wet": "[WET cough]"}.get(label, "")
    print(f"\nSegment {i} ({r['start']:.2f}s ~ {r['end']:.2f}s) — {icon} {label}")
    print(f"  {prob_str}")
    ipd.display(ipd.Audio(seg, rate=sr))

## 11. Batch Test (All v4 samples)

In [ ]:
def run_pipeline_single(audio_path):
    """Run full 2-stage pipeline on a single file. Returns result dict."""
    audio_raw, sr_raw = librosa.load(str(audio_path), sr=SR)
    segs, msk = segment_cough(audio_raw, sr_raw, cough_padding=0)
    ch = np.diff(msk.astype(int))
    ss = np.where(ch == 1)[0] + 1
    ee = np.where(ch == -1)[0] + 1

    seg_results = []
    for j, (s, e) in enumerate(zip(ss, ee)):
        if j >= len(segs):
            break
        emb = get_embedding(segs[j])

        # Stage 1
        pred_3c = clf_3c.predict(emb)[0]
        proba_3c = clf_3c.predict_proba(emb)[0]
        label_3c = le_3c.inverse_transform([pred_3c])[0]
        prob_3c = {c: round(float(p), 3) for c, p in zip(le_3c.classes_, proba_3c)}

        if label_3c == 'none':
            final_label = 'none'
            final_probs = prob_3c
        else:
            # Stage 2
            pred_2c = clf_2c.predict(emb)[0]
            proba_2c = clf_2c.predict_proba(emb)[0]
            label_2c = le_2c.inverse_transform([pred_2c])[0]
            prob_2c = {c: round(float(p), 3) for c, p in zip(le_2c.classes_, proba_2c)}
            none_p = prob_3c.get('none', 0)
            cough_p = 1.0 - none_p
            final_label = label_2c
            final_probs = {
                'none': none_p,
                'dry': round(cough_p * prob_2c.get('dry', 0), 3),
                'wet': round(cough_p * prob_2c.get('wet', 0), 3),
            }

        seg_results.append({
            'start': round(s / sr_raw, 3),
            'end': round(e / sr_raw, 3),
            'label': final_label,
            'probs': final_probs,
        })

    # Majority vote
    cough_labels = [r['label'] for r in seg_results if r['label'] in ('dry', 'wet')]
    if cough_labels:
        predicted = Counter(cough_labels).most_common(1)[0][0]
    else:
        predicted = 'none'

    return {
        'predicted': predicted,
        'n_segments': len(seg_results),
        'n_cough': len(cough_labels),
        'vote_counts': dict(Counter(cough_labels)) if cough_labels else {},
        'segments': seg_results,
    }

In [ ]:
# Run batch test on all v4 samples
input_dir = ROOT / "data" / "multimodal" / "generated_voices_v4"

files = []
for label in ("dry", "wet", "none"):
    label_dir = input_dir / label
    if label_dir.exists():
        for wav in sorted(label_dir.glob("*.wav")):
            files.append((wav, label))

print(f"Testing {len(files)} files...\n")
print(f"{'File':<40} {'GT':<6} {'Pred':<6} {'OK':<4} {'Segs':>4} {'Cough':>5} {'Votes'}")
print("-" * 90)

correct_count = 0
batch_results = []

for wav_path, gt in files:
    r = run_pipeline_single(wav_path)
    ok = r['predicted'] == gt
    if ok:
        correct_count += 1
    
    vote_str = ", ".join(f"{k}:{v}" for k, v in r['vote_counts'].items()) if r['vote_counts'] else "-"
    mark = 'O' if ok else 'X'
    print(f"{wav_path.name:<40} {gt:<6} {r['predicted']:<6} {mark:<4} {r['n_segments']:>4} {r['n_cough']:>5} {vote_str}")
    batch_results.append({**r, 'file': wav_path.name, 'gt': gt, 'correct': ok})

print(f"\nAccuracy: {correct_count}/{len(files)} ({correct_count/len(files)*100:.1f}%)")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

gt_labels = [r['gt'] for r in batch_results]
pred_labels = [r['predicted'] for r in batch_results]
labels = ['none', 'dry', 'wet']

cm = confusion_matrix(gt_labels, pred_labels, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix — v4 Pipeline Test')
plt.tight_layout()
plt.show()